In [2]:
import numpy as np
import torch
import matplotlib
from millab.src.builder import create_model
from mammoth import Mammoth
import os

matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [2]:
H5_FEAT_KEY = "feats"
COORDS_KEY = "coords"
PATCH_SIZE_AT_EXTRACTION = 256
FEATURE_EXTRACTION_MAG = 20
LEVEL0_MAG_FALLBACK = 40
MAX_THUMBNAIL_SIDE = 4000
NUM_EXPERTS = 30
NUM_SLOTS = 10
HEATMAP_ALPHA = 0.3

In [ ]:
def load_mammoth_state_dict(ckpt_path, device="cpu"):
    """
    Load checkpoint and return state_dict containing only the Mammoth submodule.
    Full MIL checkpoints often store the model under 'model' with prefix 'mlp.router.mammoth.'
    """
    ckpt = torch.load(ckpt_path, map_location=device)
    model_sd = ckpt.get("model", ckpt)
    # prefix = "mlp.router.mammoth."
    # stripped = {
    #     k[len(prefix) :]: v for k, v in model_sd.items() if k.startswith(prefix)
    # }
    # if not stripped:
    #     raise ValueError(
    #         f"No keys with prefix {prefix!r} in checkpoint. Keys: {list(model_sd.keys())[:10]}..."
    #     )
    return model_sd


def build_mammoth(ckpt_path, device="cpu"):
    """
    Build Mammoth with architecture matching the lung_tp53_abmil training setup,
    load weights from checkpoint, and return the model in eval mode.
    """
    mammoth = Mammoth(
        input_dim=1024,
        dim=512,
        num_experts=NUM_EXPERTS,
        num_slots=NUM_SLOTS,
        num_heads=16,
        lora_rank=16,
        auto_rank=False,
        slot_dim=256,
        share_lora_weights=True,
        keep_slots=True,
        dropout=0.0,
    )
    sd = load_mammoth_state_dict(ckpt_path, device)
    mammoth.load_state_dict(sd, strict=True)
    mammoth.to(device)
    mammoth.eval()
    return mammoth


def print_weight_shapes(state_dict):
    """Print shape of each tensor in the Mammoth state dict (for tutorial clarity)."""
    print("Mammoth state dict weight shapes:")
    for k, v in sorted(state_dict.items()):
        print(f"  {k}: {tuple(v.shape)}")



trial = 1
fold = 0
model_path = f"artifacts/moe_minimal_full_aug/trial_{trial}/models/fold_{fold}.pt"

build_mammoth(model_path)

In [3]:
trial = 1
fold = 0
model_path = f"artifacts/moe_minimal_full_aug/trial_{trial}/models/fold_{fold}.pt"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = create_model('abmil.base_mammoth.conch_v15', num_classes=4).to(device)

random_input = torch.randn(1, 1024, 768).to(device)

with torch.no_grad():
    output, log_dict = model(random_input, return_attention=True)
    global_attention_scores = log_dict["attention"]

"Model name abmil.base_mammoth.conch_v15 does not have a task, using default task none.
Auto-computed LoRA rank: 13 (from dimensions: input_dim=768, slot_dim=256, output_dim=512, num_experts=30)


C:\Users\gomaaad\Projects\CRC_ABMIL\millab\src\builders\ModelDict.py:193: UserWarning: Pretrained flag is True, but task is set to 'none'. Using random weights
  warnings.warn("Pretrained flag is True, but task is set to 'none'. Using random weights")
